# RHI Live Runtime v12 — Grounded Recursive Branch Agent

Δ Purpose: v11 proved the controller can run, but the uploaded outputs show a new failure mode:

$$
\text{all prompts} \rightarrow \Psi_{\text{direct margin}} \quad \text{at depth }0
$$

That is not enough. The runtime must distinguish:

$$
\text{generic controller boilerplate} \neq \text{prompt-grounded operational answer}
$$

v12 adds the missing gate:

$$
Q \rightarrow C_Q \rightarrow B_i \rightarrow A_i \rightarrow G_{\text{grounded}} \rightarrow \Psi \text{ or } \Omega
$$

where $G_{\text{grounded}}$ requires prompt specificity, low boilerplate, trace sufficiency, and contract stance.

## What v12 fixes

1. **Records generation provenance**: model output vs deterministic fallback.
2. **No silent fallback illusion**: if the model fails, the output says so.
3. **Blocks generic construct answers**: direct margin alone cannot collapse.
4. **Adds prompt-grounding score**: the answer must fit this prompt, not any Nexus prompt.
5. **Adds boilerplate penalty**: repeated scaffold phrases weaken collapse.
6. **Adds branch-leak detection**: a branch answering the wrong prompt is punished.
7. **Makes recursion visible**: failed dimensions create $\Omega$, then repair the contract and regenerate.
8. **Keeps paths local**: current notebook folder is root; no `/mnt/data`, no fake project paths.

## Read this first

Run cells top to bottom. Outputs save into:

```text
./rhi_v12_outputs/
```


In [1]:
# Optional install cell. Run only if your environment is missing packages.
# %pip install -U pandas numpy torch transformers accelerate safetensors sentencepiece

from __future__ import annotations

import os
import re
import json
import math
import time
import uuid
import random
import hashlib
import traceback
from dataclasses import dataclass, asdict, field, replace
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v12_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v12_" + uuid.uuid4().hex[:10]
SEED = 11

random.seed(SEED)
np.random.seed(SEED)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("RUN_ID:", RUN_ID)


ROOT: D:\@User Data\Downloads
OUT_DIR: D:\@User Data\Downloads\rhi_v12_outputs
RUN_ID: rhi_v12_a551b6e41e


## 1. Model loading

v11 said `MODEL_READY=True`, but the run logs show `AttributeError` during generation and every branch fell back to deterministic text.

v12 makes that impossible to miss. Every candidate carries:

```text
origin = model | fallback
error = None | exception text
```

That is important because fallback branches are useful for controller testing, but they are **not** proof that the live model is participating.


In [2]:
# Model configuration.
# Put a downloaded model folder next to this notebook, or set RHI_MODEL manually.
#
# Examples:
#   MODEL_ID_OR_PATH = "./Qwen2.5-1.5B-Instruct"
#   MODEL_ID_OR_PATH = "C:/Users/Dean/Downloads/Qwen2.5-1.5B-Instruct"
#   MODEL_ID_OR_PATH = "Qwen/Qwen2.5-1.5B-Instruct"

default_local = "./Qwen2.5-1.5B-Instruct" if Path("./Qwen2.5-1.5B-Instruct").exists() else "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", default_local)

LOAD_REAL_MODEL = True
MAX_NEW_TOKENS = 420
TEMPERATURE = 0.55
TOP_P = 0.90

tokenizer = None
model = None
MODEL_READY = False
DEVICE_INFO: Dict[str, Any] = {}
MODEL_ERROR: Optional[str] = None

def try_load_model(model_id_or_path: str) -> bool:
    global tokenizer, model, MODEL_READY, DEVICE_INFO, MODEL_ERROR

    if not LOAD_REAL_MODEL:
        MODEL_READY = False
        MODEL_ERROR = "LOAD_REAL_MODEL=False"
        print("Model loading disabled. Using fallback branches.")
        return False

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        DEVICE_INFO["torch_version"] = torch.__version__
        DEVICE_INFO["cuda_available"] = bool(torch.cuda.is_available())
        DEVICE_INFO["device_count"] = int(torch.cuda.device_count())
        if torch.cuda.is_available():
            DEVICE_INFO["gpu_name"] = torch.cuda.get_device_name(0)
            DEVICE_INFO["cuda_version"] = torch.version.cuda

        print("Torch/CUDA:", DEVICE_INFO)

        hf_token = os.environ.get("HF_TOKEN", None)
        tokenizer = AutoTokenizer.from_pretrained(
            model_id_or_path,
            trust_remote_code=True,
            token=hf_token,
        )

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32

        # Newer transformers prefers `dtype`; older accepts `torch_dtype`.
        try:
            model = AutoModelForCausalLM.from_pretrained(
                model_id_or_path,
                trust_remote_code=True,
                dtype=dtype,
                device_map="auto" if torch.cuda.is_available() else None,
                low_cpu_mem_usage=True,
                token=hf_token,
            )
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(
                model_id_or_path,
                trust_remote_code=True,
                torch_dtype=dtype,
                device_map="auto" if torch.cuda.is_available() else None,
                low_cpu_mem_usage=True,
                token=hf_token,
            )

        model.eval()
        MODEL_READY = True
        MODEL_ERROR = None
        print("MODEL_READY:", MODEL_READY)
        print("MODEL_ID_OR_PATH:", model_id_or_path)
        return True

    except Exception as e:
        MODEL_READY = False
        MODEL_ERROR = type(e).__name__ + ": " + str(e)
        print("MODEL LOAD FAILED — using fallback branches.")
        print(MODEL_ERROR)
        return False

_ = try_load_model(MODEL_ID_OR_PATH)


Torch/CUDA: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'gpu_name': 'NVIDIA GeForce RTX 4060', 'cuda_version': '12.6'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

MODEL_READY: True
MODEL_ID_OR_PATH: Qwen/Qwen2.5-1.5B-Instruct


## 2. Core text and shape functions

v12 splits four things that were blended before:

$$
\text{surface vocabulary},\quad \text{prompt grounding},\quad \text{contract stance},\quad \text{operational fit}
$$

A branch must now satisfy all four enough to collapse.


In [3]:
STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","of","to","in","on","for","with","by","as",
    "is","are","was","were","be","being","been","it","this","that","these","those","from","at",
    "into","out","about","so","because","therefore","than","not","no","yes","do","does","did",
    "can","could","should","would","will","just","they","them","their","you","your","we","our",
    "i","me","my","he","she","his","her","its","when","where","what","why","how"
}

NEXUS_SURFACE_TERMS = {
    "nexus","contract","carrier","domain","boundary","collapse","shape","value","slot","need",
    "forbidden","neighbor","operational","recursive","recursion","krrb","omega","psi","field",
    "fold","runtime","phase","lock","audit","trace","signal","evidence","branch","repair",
    "candidate","construct","verify","gate","hot","cold","preserve","function"
}

BOILERPLATE_PHRASES = [
    "the prompt is asking",
    "forms a need-slot before acting",
    "correct flow is prompt",
    "prompt → contract",
    "branch candidates",
    "operational audit",
    "gated collapse",
    "answer should not be a noun lookup",
    "construct the inverse shape",
    "missing operational slot implied by the prompt",
]

TOOL_LEAK_PHRASES = [
    "current agents fail when they call tools",
    "tool output becomes the driver",
    "tool-first execution looks efficient",
    "tools as constrained evidence channels",
]

def words(text: str, remove_nexus_surface: bool = False) -> List[str]:
    toks = re.findall(r"[a-zA-Z0-9_ΔΨΩ]+", str(text).lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    if remove_nexus_surface:
        toks = [t for t in toks if t not in NEXUS_SURFACE_TERMS]
    return toks

def wordset(text: str, remove_nexus_surface: bool = False) -> set:
    return set(words(text, remove_nexus_surface=remove_nexus_surface))

def jaccard_text(a: str, b: str, remove_nexus_surface: bool = False) -> float:
    wa = wordset(a, remove_nexus_surface=remove_nexus_surface)
    wb = wordset(b, remove_nexus_surface=remove_nexus_surface)
    if not wa and not wb:
        return 1.0
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / max(1, len(wa | wb))

def contains_any(text: str, terms: List[str]) -> bool:
    s = str(text).lower()
    return any(str(t).lower() in s for t in terms)

def phrase_count(text: str, phrases: List[str]) -> int:
    s = str(text).lower()
    return sum(1 for p in phrases if p in s)

def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, float(x)))

def harmonic_mean(vals: List[float], eps: float = 1e-9) -> float:
    vals = [max(eps, float(v)) for v in vals]
    return len(vals) / sum(1.0 / v for v in vals)

def safe_mean(vals: List[float]) -> float:
    return float(np.mean(vals)) if vals else 0.0


## 3. Shape templates and need-slot contract

The prompt becomes an inverse cavity:

$$
C_Q = (N_Q, F_Q, B_Q, T_Q, \Psi_Q)
$$

But v12 adds a stricter idea:

$$
C_Q \neq \text{generic framework wrapper}
$$

The contract must carry prompt-specific terms and verbs.


In [4]:
SHAPE_TEMPLATES = {
    "CONTRACT": {
        "triggers": ["contract", "intent", "before", "plan", "spec", "interface"],
        "needs": ["intent", "boundary", "before", "select", "gate", "success"],
    },
    "TOOL": {
        "triggers": ["tool", "api", "call", "function", "agent", "execute", "act"],
        "needs": ["tool", "input", "output", "side-effect", "verify", "evidence"],
    },
    "GROOVE": {
        "triggers": ["train", "lora", "qlora", "adapter", "fine tune", "weights", "groove", "model"],
        "needs": ["adapter", "low-rank", "weights", "delta", "dataset", "loss", "eval", "base"],
    },
    "SEARCH": {
        "triggers": ["search", "retrieve", "find", "query", "lookup", "index", "rag"],
        "needs": ["query", "retrieve", "candidate", "rank", "verify", "evidence", "no noun"],
    },
    "REPAIR": {
        "triggers": ["fix", "repair", "error", "failed", "broken", "bug", "traceback", "syntaxerror", "nameerror"],
        "needs": ["failure", "cause", "patch", "test", "rerun", "trace"],
    },
    "MEMORY": {
        "triggers": ["remember", "memory", "recall", "lost", "state", "context", "summary", "continuity"],
        "needs": ["state", "trace", "retrieve", "preserve", "update", "continuity"],
    },
    "BOUNDARY": {
        "triggers": ["boundary", "limit", "forbidden", "constraint", "safety", "gate", "reject"],
        "needs": ["boundary", "reject", "constraint", "preserve", "violate", "gate"],
    },
    "RECURSE": {
        "triggers": ["recursive", "recursion", "again", "loop", "fold", "iterate", "turn", "depth"],
        "needs": ["recursive", "branch", "feedback", "repair", "collapse", "omega", "rerun"],
    },
}

ACTION_VERBS = {
    "explain","fix","repair","design","train","retrieve","search","form","preserve","compare","build","generate",
    "verify","collapse","recurse","overwriting","overwrite","remember","summarize","continue","use","fail"
}

def detect_shape_template(prompt: str) -> List[str]:
    p = str(prompt).lower()
    active = []
    for name, cfg in SHAPE_TEMPLATES.items():
        hits = sum(1 for t in cfg["triggers"] if t in p)
        if hits:
            active.append(name)
    return active or ["GENERAL"]

def extract_domain_terms(prompt: str, max_terms: int = 14) -> List[str]:
    ws = words(prompt, remove_nexus_surface=True)
    counts = Counter(ws)
    return [w for w, _ in counts.most_common(max_terms)]

def extract_action_terms(prompt: str) -> List[str]:
    ws = words(prompt, remove_nexus_surface=True)
    found = [w for w in ws if w in ACTION_VERBS or w.endswith("ing") or w.endswith("ed")]
    # Keep unique order.
    seen = set()
    out = []
    for w in found:
        if w not in seen:
            out.append(w)
            seen.add(w)
    return out[:8]

def infer_forbidden_neighbors(prompt: str, active: List[str]) -> List[str]:
    forb = set(["generic controller boilerplate", "surface label"])
    if "TOOL" in active or "CONTRACT" in active:
        forb.update(["tool-first action", "premature execution", "api reflex", "surface task completion"])
    if "GROOVE" in active:
        forb.update(["full retrain reflex", "weight-churn", "dataset worship", "loss-only tuning"])
    if "SEARCH" in active:
        forb.update(["noun lookup", "keyword matching", "unverified retrieval", "search without verifier"])
    if "REPAIR" in active:
        forb.update(["blanket rewrite", "threshold fiddling", "silent failure", "patch without test"])
    if "MEMORY" in active:
        forb.update(["stateless answer", "context amnesia", "surface recall", "summary as memory"])
    if "BOUNDARY" in active:
        forb.update(["unsafe override", "constraint erasure", "boundary confusion"])
    if "RECURSE" in active:
        forb.update(["linear pipeline", "single branch", "dead loop", "nested sweep masquerading as recursion"])
    if active == ["GENERAL"]:
        forb.update(["noun-only answer", "generic explanation"])
    return sorted(forb)

@dataclass
class NeedSlotContract:
    prompt: str
    active_templates: List[str]
    inverse_need: str
    preserved_function: str
    boundary_conditions: List[str]
    domain_carrier: List[str]
    action_terms: List[str]
    forbidden_neighbors: List[str]
    collapse_target: str
    repair_history: List[Dict[str, Any]] = field(default_factory=list)

def build_contract(prompt: str) -> NeedSlotContract:
    active = detect_shape_template(prompt)
    domain_terms = extract_domain_terms(prompt)
    action_terms = extract_action_terms(prompt)

    inverse_need = (
        "build the missing operational shape demanded by this prompt; "
        "answer this prompt specifically; reject generic framework boilerplate"
    )

    preserved_parts = []
    if "TOOL" in active or "CONTRACT" in active:
        preserved_parts.append("form the contract before tool/action selection")
    if "GROOVE" in active:
        preserved_parts.append("train a slot-builder through low-rank delta without overwriting the base model")
    if "SEARCH" in active:
        preserved_parts.append("retrieve by inverse shape when noun match is absent")
    if "REPAIR" in active:
        preserved_parts.append("repair the failed operational dimension and rerun recursively")
    if "MEMORY" in active:
        preserved_parts.append("preserve trace continuity rather than substituting a text summary")
    if "RECURSE" in active:
        preserved_parts.append("branch recursively until grounded Ψ or explicit Ω residue")
    if not preserved_parts:
        preserved_parts.append("preserve the prompt's verb-level operation")

    boundary_conditions = [
        "do not collapse on shared Nexus vocabulary alone",
        "do not collapse on generic controller boilerplate",
        "require prompt-specific domain carriers in the answer",
        "require evidence trace for the selected answer",
        "prefer Ω over false Ψ when top branches disagree operationally",
        "preserve base answer when controller evidence is weak",
    ]

    collapse_target = (
        "one prompt-specific executable answer with contract fit, prompt grounding, "
        "operational evidence, and trace sufficient to debug"
    )

    return NeedSlotContract(
        prompt=prompt,
        active_templates=active,
        inverse_need=inverse_need,
        preserved_function="; ".join(preserved_parts),
        boundary_conditions=boundary_conditions,
        domain_carrier=domain_terms,
        action_terms=action_terms,
        forbidden_neighbors=infer_forbidden_neighbors(prompt, active),
        collapse_target=collapse_target,
    )

def contract_to_text(c: NeedSlotContract) -> str:
    return (
        f"ACTIVE_TEMPLATES: {', '.join(c.active_templates)}\n"
        f"INVERSE_NEED: {c.inverse_need}\n"
        f"PRESERVED_FUNCTION: {c.preserved_function}\n"
        f"DOMAIN_CARRIER: {', '.join(c.domain_carrier)}\n"
        f"ACTION_TERMS: {', '.join(c.action_terms)}\n"
        f"FORBIDDEN_NEIGHBORS: {' | '.join(c.forbidden_neighbors)}\n"
        f"BOUNDARY_CONDITIONS: {' | '.join(c.boundary_conditions)}\n"
        f"COLLAPSE_TARGET: {c.collapse_target}\n"
        f"REPAIR_HISTORY: {json.dumps(c.repair_history, ensure_ascii=False)}"
    )


## 4. Candidate branches

The branch prompts now force **actual answer content**, not controller self-description.

v11 fallback produced the same scaffold across prompts. v12 tracks and penalizes that.


In [5]:
BRANCH_SYSTEMS = {
    "construct": (
        "You are the CONSTRUCT branch. Answer the user's prompt directly from the need-slot contract. "
        "Do not describe the controller unless the prompt asks about the controller. "
        "Use concrete mechanisms, not labels."
    ),
    "verify": (
        "You are the VERIFY branch. Test the likely answer against the contract. "
        "Name what would make the answer fail. Prefer operational disagreement over vocabulary agreement."
    ),
    "repair": (
        "You are the REPAIR branch. If the answer would fail, patch only the failed dimension. "
        "Do not blanket rewrite. Produce the corrected operational answer."
    ),
    "counter": (
        "You are the COUNTER branch. Identify the strongest wrong path and why it fails. "
        "Then provide the corrected path."
    ),
}

def fallback_branch(prompt: str, contract: NeedSlotContract, branch_name: str) -> str:
    # Prompt-grounded fallback. This is still not model proof, but it lets the controller run.
    active = ", ".join(contract.active_templates)
    domain = ", ".join(contract.domain_carrier[:8])
    actions = ", ".join(contract.action_terms[:5]) if contract.action_terms else "preserve"
    preserved = contract.preserved_function

    if branch_name == "construct":
        return (
            f"Δ Operational answer for this prompt: active={active}. "
            f"The missing shape is not a noun; it is the operation `{preserved}`. "
            f"Use carriers [{domain}] and actions [{actions}] as the fit test. "
            f"The answer must perform that operation, reject {contract.forbidden_neighbors[0]}, "
            "and leave a trace showing why the selected path fits."
        )
    if branch_name == "verify":
        return (
            f"Verification: test whether the candidate actually preserves `{preserved}`. "
            f"Check prompt carriers [{domain}], then check boundary, trap rejection, and collapse target. "
            "If the answer only repeats framework words, mark Ω_boilerplate instead of Ψ."
        )
    if branch_name == "repair":
        return (
            f"Repair: if prompt grounding is weak, inject the missing carriers [{domain}] and the preserved operation `{preserved}`. "
            "If boundary is weak, add the constraint it must not violate. "
            "If collapse is weak, state the executable output and rerun the gate."
        )
    if branch_name == "counter":
        return (
            f"Wrong path: answering with a generic contract scaffold ignores the prompt's own carriers [{domain}]. "
            f"Correct path: use the inverse need to produce an answer that does `{preserved}` and nothing broader."
        )
    return "No branch."

def _apply_chat_template(messages):
    # Some tokenizer/model combos throw AttributeError inside apply_chat_template in certain envs.
    # v12 isolates that failure and records it.
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        )
    return None

def model_generate_one(prompt: str, contract: NeedSlotContract, branch_name: str) -> Dict[str, Any]:
    if not MODEL_READY:
        return {
            "branch": branch_name,
            "answer": fallback_branch(prompt, contract, branch_name),
            "origin": "fallback",
            "error": MODEL_ERROR or "MODEL_READY=False",
        }

    import torch

    system = BRANCH_SYSTEMS[branch_name]
    user = (
        "USER_PROMPT:\n" + prompt.strip() + "\n\n"
        "NEED_SLOT_CONTRACT:\n" + contract_to_text(contract) + "\n\n"
        "Rules:\n"
        "1. Answer this exact prompt, not a generic RHI prompt.\n"
        "2. Do not use the phrase 'the prompt is asking'.\n"
        "3. Do not output 'prompt → contract → branch candidates' as the answer.\n"
        "4. Include a concrete mechanism, a failure mode, and a check.\n"
        "5. Keep it compact.\n\n"
        "ANSWER:\n"
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    try:
        if hasattr(tokenizer, "apply_chat_template"):
            input_ids = _apply_chat_template(messages)
            if input_ids is None:
                raise AttributeError("tokenizer.apply_chat_template returned None")
            # Important: do not assume model.device exists under device_map='auto'.
            device = next(model.parameters()).device
            input_ids = input_ids.to(device)
            with torch.no_grad():
                out = model.generate(
                    input_ids,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    pad_token_id=tokenizer.eos_token_id,
                )
            gen = out[0][input_ids.shape[-1]:]
            text = tokenizer.decode(gen, skip_special_tokens=True).strip()
        else:
            text_in = system + "\n\n" + user
            device = next(model.parameters()).device
            inputs = tokenizer(text_in, return_tensors="pt").to(device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    pad_token_id=tokenizer.eos_token_id,
                )
            text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

        if not text:
            raise ValueError("empty model generation")

        return {"branch": branch_name, "answer": text, "origin": "model", "error": None}

    except Exception as e:
        return {
            "branch": branch_name,
            "answer": fallback_branch(prompt, contract, branch_name),
            "origin": "fallback",
            "error": type(e).__name__ + ": " + str(e),
        }

def generate_candidates(prompt: str, contract: NeedSlotContract) -> List[Dict[str, Any]]:
    return [model_generate_one(prompt, contract, b) for b in BRANCH_SYSTEMS]


## 5. Operational audit

v12 adds these observables:

$$
A_i = (F_{\text{need}}, F_{\text{function}}, F_{\text{boundary}}, F_{\text{trap}}, F_{\text{collapse}}, F_{\text{prompt}}, F_{\text{boilerplate}})
$$

The new terms matter:

- $F_{\text{prompt}}$: does the answer fit this exact prompt?
- $F_{\text{boilerplate}}$: does it avoid reusable scaffold text?
- $F_{\text{leak}}$: did a branch answer a different prompt family?


In [6]:
def field_hit_score(text: str, field_terms: List[str], cap: int = 8) -> float:
    if not field_terms:
        return 0.5
    s = str(text).lower()
    clean_terms = [str(t).lower() for t in field_terms if str(t).strip()]
    hits = 0
    for term in clean_terms[:cap]:
        if term in s:
            hits += 1
    return clamp(hits / max(1, min(len(clean_terms), cap)))

def contract_field_terms(contract: NeedSlotContract) -> Dict[str, List[str]]:
    return {
        "need": words(contract.inverse_need, remove_nexus_surface=True),
        "function": words(contract.preserved_function, remove_nexus_surface=True),
        "boundary": words(" ".join(contract.boundary_conditions), remove_nexus_surface=True),
        "domain": contract.domain_carrier,
        "actions": contract.action_terms,
        "forbidden": words(" ".join(contract.forbidden_neighbors), remove_nexus_surface=True),
        "collapse": words(contract.collapse_target, remove_nexus_surface=True),
    }

def shape_template_score(text: str, active_templates: List[str]) -> float:
    if active_templates == ["GENERAL"]:
        return 0.5
    vals = []
    for t in active_templates:
        needs = SHAPE_TEMPLATES.get(t, {}).get("needs", [])
        vals.append(field_hit_score(text, needs, cap=len(needs)))
    return clamp(safe_mean(vals))

def prompt_grounding_score(prompt: str, contract: NeedSlotContract, answer: str) -> Dict[str, float]:
    fields = contract_field_terms(contract)
    domain = field_hit_score(answer, fields["domain"], cap=10)
    actions = field_hit_score(answer, fields["actions"], cap=8)
    function = field_hit_score(answer, fields["function"], cap=10)
    raw_prompt_overlap = jaccard_text(prompt, answer, remove_nexus_surface=True)

    # Prompt grounding must not be pure overlap; it should include action/function.
    prompt_fit = clamp(0.40 * domain + 0.25 * actions + 0.25 * function + 0.10 * raw_prompt_overlap)

    return {
        "domain_hit": domain,
        "action_hit": actions,
        "function_hit": function,
        "raw_prompt_overlap": raw_prompt_overlap,
        "prompt_fit": prompt_fit,
    }

def boilerplate_scores(prompt: str, contract: NeedSlotContract, answer: str) -> Dict[str, float]:
    bp_count = phrase_count(answer, BOILERPLATE_PHRASES)
    boilerplate_penalty = clamp(bp_count / 4.0)
    anti_boilerplate = clamp(1.0 - boilerplate_penalty)

    # Tool leakage only counts as leakage when the prompt is not about tools/contracts.
    active = set(contract.active_templates)
    tool_leak_count = phrase_count(answer, TOOL_LEAK_PHRASES)
    if ("TOOL" not in active) and ("CONTRACT" not in active):
        branch_leak_penalty = clamp(tool_leak_count / 2.0)
    else:
        branch_leak_penalty = 0.0

    return {
        "boilerplate_count": float(bp_count),
        "boilerplate_penalty": boilerplate_penalty,
        "anti_boilerplate": anti_boilerplate,
        "branch_leak_count": float(tool_leak_count),
        "branch_leak_penalty": branch_leak_penalty,
    }

def answer_operational_audit(prompt: str, contract: NeedSlotContract, answer: str) -> Dict[str, Any]:
    fields = contract_field_terms(contract)
    a = str(answer).lower()

    pg = prompt_grounding_score(prompt, contract, answer)
    bp = boilerplate_scores(prompt, contract, answer)

    F_need = clamp(
        0.45 * field_hit_score(answer, fields["need"]) +
        0.35 * pg["domain_hit"] +
        0.20 * pg["action_hit"]
    )

    F_function = clamp(
        0.60 * field_hit_score(answer, fields["function"]) +
        0.25 * pg["action_hit"] +
        0.15 * sum([
            contains_any(a, ["preserve", "maintain", "continue", "function", "operation"]),
            contains_any(a, ["mechanism", "step", "runtime", "select", "verify", "repair", "train", "retrieve"]),
        ]) / 2
    )

    F_boundary = clamp(
        0.60 * field_hit_score(answer, fields["boundary"]) +
        0.40 * sum([
            contains_any(a, ["boundary", "constraint", "gate", "reject", "protect", "forbidden"]),
            contains_any(a, ["false", "wrong", "weak", "unsafe", "premature", "surface", "generic"]),
        ]) / 2
    )

    forbidden_hit = field_hit_score(answer, fields["forbidden"])
    trap_language = sum([
        contains_any(a, ["not", "instead", "wrong", "fails", "reject", "avoid", "forbidden"]),
        contains_any(a, ["surface", "generic", "threshold", "noun", "keyword", "boilerplate"]),
    ]) / 2
    F_trap = clamp(0.45 * forbidden_hit + 0.55 * trap_language)

    F_collapse = clamp(
        0.45 * field_hit_score(answer, fields["collapse"]) +
        0.30 * contains_any(a, ["therefore", "so", "because", "result", "collapse", "answer"]) +
        0.25 * contains_any(a, ["one", "single", "executable", "run", "test", "trace", "check"])
    )

    F_shape = shape_template_score(answer, contract.active_templates)

    # New v12 field: prompt specificity.
    F_prompt = pg["prompt_fit"]

    hot = clamp((F_need + F_function + F_shape + F_prompt) / 4)
    cold = clamp((F_boundary + F_trap + F_collapse + bp["anti_boilerplate"]) / 4)
    hotcold_balance = clamp(1.0 - abs(hot - cold))

    quality_core = harmonic_mean([F_need, F_function, F_boundary, F_trap, F_collapse])
    quality_grounded = harmonic_mean([
        F_need, F_function, F_boundary, F_trap, F_collapse,
        max(1e-9, F_prompt),
        max(1e-9, bp["anti_boilerplate"]),
    ])

    return {
        "F_need": F_need,
        "F_function": F_function,
        "F_boundary": F_boundary,
        "F_trap": F_trap,
        "F_collapse": F_collapse,
        "F_shape": F_shape,
        "F_prompt": F_prompt,
        "hot": hot,
        "cold": cold,
        "hotcold_balance": hotcold_balance,
        "quality_core_hmean": quality_core,
        "quality_grounded_hmean": quality_grounded,
        **pg,
        **bp,
    }

def audit_agreement(audit_a: Dict[str, Any], audit_b: Dict[str, Any]) -> float:
    keys = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse", "F_prompt"]
    diffs = [abs(float(audit_a[k]) - float(audit_b[k])) for k in keys]
    return clamp(1.0 - float(np.mean(diffs)))

def contract_stance_agreement(contract: NeedSlotContract, answer_a: str, answer_b: str) -> Dict[str, Any]:
    fields = contract_field_terms(contract)
    scores = {}
    for field_name in ["need", "function", "boundary", "domain", "actions", "collapse"]:
        terms = fields[field_name]
        sa = set(t for t in terms if str(t).lower() in str(answer_a).lower())
        sb = set(t for t in terms if str(t).lower() in str(answer_b).lower())
        if not terms:
            score = 0.5
        elif not sa and not sb:
            score = 0.15
        else:
            score = len(sa & sb) / max(1, len(sa | sb))
        scores[field_name] = clamp(score)

    # Forbidden stance: both should reject similar trap families.
    fterms = fields["forbidden"]
    fa = set(t for t in fterms if str(t).lower() in str(answer_a).lower())
    fb = set(t for t in fterms if str(t).lower() in str(answer_b).lower())
    scores["forbidden"] = clamp(len(fa & fb) / max(1, len(fa | fb))) if (fa or fb) else 0.15

    aggregate = harmonic_mean(list(scores.values()))
    return {"aggregate": aggregate, "fields": scores}


## 6. Score

v12 score:

$$
S_i =
0.24 Q_i +
0.18 K_i +
0.18 P_i +
0.14 T_i +
0.10 R_i +
0.10 B_i +
0.06 O_i
-
0.18 L_i
$$

where:

- $Q_i$ = grounded operational quality
- $K_i$ = contract stance
- $P_i$ = prompt grounding
- $T_i$ = trace sufficiency
- $R_i$ = hot/cold balance
- $B_i$ = anti-boilerplate
- $O_i$ = real model origin bonus
- $L_i$ = wrong-prompt leakage


In [7]:
def trace_sufficiency(answer: str, audit: Dict[str, Any], contract: NeedSlotContract) -> float:
    a = str(answer).lower()
    bits = [
        contains_any(a, ["because", "therefore", "so", "why", "means"]),
        contains_any(a, ["boundary", "constraint", "failure", "wrong", "reject"]),
        contains_any(a, ["verify", "evidence", "trace", "audit", "test", "check"]),
        contains_any(a, ["mechanism", "step", "runtime", "operation", "train", "retrieve", "memory"]),
        audit["quality_grounded_hmean"] >= 0.36,
    ]
    return clamp(sum(bits) / len(bits))

def branch_score(prompt: str, contract: NeedSlotContract, branch: Dict[str, Any]) -> Dict[str, Any]:
    answer = branch["answer"]
    audit = answer_operational_audit(prompt, contract, answer)
    fields = contract_field_terms(contract)

    contract_stance = safe_mean([
        field_hit_score(answer, fields["need"]),
        field_hit_score(answer, fields["function"]),
        field_hit_score(answer, fields["boundary"]),
        field_hit_score(answer, fields["domain"]),
        field_hit_score(answer, fields["actions"]) if fields["actions"] else 0.5,
        field_hit_score(answer, fields["collapse"]),
    ])

    trace = trace_sufficiency(answer, audit, contract)
    origin_bonus = 1.0 if branch.get("origin") == "model" else 0.25

    score = clamp(
        0.24 * audit["quality_grounded_hmean"] +
        0.18 * contract_stance +
        0.18 * audit["F_prompt"] +
        0.14 * trace +
        0.10 * audit["hotcold_balance"] +
        0.10 * audit["anti_boilerplate"] +
        0.06 * origin_bonus -
        0.18 * audit["branch_leak_penalty"]
    )

    return {
        **branch,
        "score": score,
        "contract_stance": float(contract_stance),
        "trace_sufficiency": trace,
        "origin_bonus": origin_bonus,
        "audit": audit,
    }

def score_candidates(prompt: str, contract: NeedSlotContract, candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = [branch_score(prompt, contract, c) for c in candidates]
    flat = []
    for r in rows:
        audit = r["audit"]
        flat.append({
            "branch": r["branch"],
            "origin": r.get("origin"),
            "score": r["score"],
            "contract_stance": r["contract_stance"],
            "trace_sufficiency": r["trace_sufficiency"],
            "origin_bonus": r["origin_bonus"],
            "quality_grounded_hmean": audit["quality_grounded_hmean"],
            "quality_core_hmean": audit["quality_core_hmean"],
            "F_need": audit["F_need"],
            "F_function": audit["F_function"],
            "F_boundary": audit["F_boundary"],
            "F_trap": audit["F_trap"],
            "F_collapse": audit["F_collapse"],
            "F_shape": audit["F_shape"],
            "F_prompt": audit["F_prompt"],
            "domain_hit": audit["domain_hit"],
            "action_hit": audit["action_hit"],
            "function_hit": audit["function_hit"],
            "anti_boilerplate": audit["anti_boilerplate"],
            "boilerplate_count": audit["boilerplate_count"],
            "branch_leak_penalty": audit["branch_leak_penalty"],
            "hot": audit["hot"],
            "cold": audit["cold"],
            "hotcold_balance": audit["hotcold_balance"],
            "error": r.get("error"),
            "answer": r["answer"],
            "detail": r,
        })
    df = pd.DataFrame(flat).sort_values("score", ascending=False).reset_index(drop=True)
    return df


## 7. Collapse gates

The v11 gate was:

$$
S_{\text{top}} \ge \Psi_{\min} \quad \wedge \quad \Delta S \ge m_{\min}
$$

v12 adds:

$$
F_{\text{prompt}} \ge p_{\min}
\quad\wedge\quad
F_{\text{boilerplate}} \ge b_{\min}
\quad\wedge\quad
L_{\text{leak}} = 0
$$

So margin alone cannot collapse.


In [8]:
MAX_RECURSION_DEPTH = 3

PSI_MIN = 0.58
DIRECT_MARGIN_MIN = 0.070
PROMPT_FIT_MIN = 0.34
ANTI_BOILERPLATE_MIN = 0.55
TRACE_MIN = 0.55
QUALITY_MIN = 0.34
LEAK_MAX = 0.0

# When LOAD_REAL_MODEL=True, a fallback candidate may be useful for debugging,
# but it should not count as a real Ψ collapse.
REQUIRE_MODEL_ORIGIN_FOR_PSI = True

CONSENSUS_MARGIN_MAX = 0.055
CONSENSUS_STANCE_MIN = 0.22
CONSENSUS_AUDIT_MIN = 0.74
CONSENSUS_PROMPT_MIN = 0.34

def direct_collapse_gate(score_df: pd.DataFrame) -> Dict[str, Any]:
    top = score_df.iloc[0]
    margin = float(top["score"] - score_df.iloc[1]["score"]) if len(score_df) > 1 else float(top["score"])

    checks = {
        "score_ok": float(top["score"]) >= PSI_MIN,
        "margin_ok": margin >= DIRECT_MARGIN_MIN,
        "prompt_fit_ok": float(top["F_prompt"]) >= PROMPT_FIT_MIN,
        "anti_boilerplate_ok": float(top["anti_boilerplate"]) >= ANTI_BOILERPLATE_MIN,
        "trace_ok": float(top["trace_sufficiency"]) >= TRACE_MIN,
        "quality_ok": float(top["quality_grounded_hmean"]) >= QUALITY_MIN,
        "leak_ok": float(top["branch_leak_penalty"]) <= LEAK_MAX,
        "origin_ok": (not REQUIRE_MODEL_ORIGIN_FOR_PSI) or (not LOAD_REAL_MODEL) or str(top.get("origin", "")) == "model",
    }
    ok = all(checks.values())

    reason = "direct_grounded_collapse" if ok else "no_direct_collapse"
    return {
        "ok": bool(ok),
        "reason": reason,
        "margin": margin,
        "top_score": float(top["score"]),
        "top_branch": str(top["branch"]),
        "checks": checks,
    }

def consensus_gate(contract: NeedSlotContract, score_df: pd.DataFrame) -> Dict[str, Any]:
    if len(score_df) < 2:
        return {"ok": False, "reason": "not_enough_branches"}

    a = score_df.iloc[0]["detail"]
    b = score_df.iloc[1]["detail"]
    margin = float(score_df.iloc[0]["score"] - score_df.iloc[1]["score"])

    lex_plain = jaccard_text(a["answer"], b["answer"], remove_nexus_surface=False)
    lex_operational = jaccard_text(a["answer"], b["answer"], remove_nexus_surface=True)
    audit_ag = audit_agreement(a["audit"], b["audit"])
    stance = contract_stance_agreement(contract, a["answer"], b["answer"])

    both_high = (
        float(score_df.iloc[0]["score"]) >= PSI_MIN and
        float(score_df.iloc[1]["score"]) >= PSI_MIN * 0.92 and
        float(score_df.iloc[0]["F_prompt"]) >= CONSENSUS_PROMPT_MIN and
        float(score_df.iloc[1]["F_prompt"]) >= CONSENSUS_PROMPT_MIN and
        float(score_df.iloc[0]["anti_boilerplate"]) >= ANTI_BOILERPLATE_MIN and
        float(score_df.iloc[1]["anti_boilerplate"]) >= ANTI_BOILERPLATE_MIN and
        ((not REQUIRE_MODEL_ORIGIN_FOR_PSI) or (not LOAD_REAL_MODEL) or (str(score_df.iloc[0]["origin"]) == "model" and str(score_df.iloc[1]["origin"]) == "model"))
    )

    ok = (
        both_high and
        margin <= CONSENSUS_MARGIN_MAX and
        audit_ag >= CONSENSUS_AUDIT_MIN and
        stance["aggregate"] >= CONSENSUS_STANCE_MIN
    )

    return {
        "ok": bool(ok),
        "reason": "contract_anchored_consensus" if ok else "no_consensus",
        "margin": margin,
        "both_high": bool(both_high),
        "lex_plain": lex_plain,
        "lex_operational": lex_operational,
        "audit_agreement": audit_ag,
        "stance_agreement": stance["aggregate"],
        "stance_fields": stance["fields"],
    }

def extract_omega(score_df: pd.DataFrame, contract: NeedSlotContract) -> Dict[str, Any]:
    top = score_df.iloc[0]
    dimensions = {
        "need": float(top["F_need"]),
        "function": float(top["F_function"]),
        "boundary": float(top["F_boundary"]),
        "trap": float(top["F_trap"]),
        "collapse": float(top["F_collapse"]),
        "prompt": float(top["F_prompt"]),
        "anti_boilerplate": float(top["anti_boilerplate"]),
        "trace": float(top["trace_sufficiency"]),
        "quality": float(top["quality_grounded_hmean"]),
    }
    weakest = sorted(dimensions.items(), key=lambda kv: kv[1])[:3]

    gate_failures = []
    direct = direct_collapse_gate(score_df)
    for k, v in direct["checks"].items():
        if not v:
            gate_failures.append(k)

    return {
        "winner_branch": str(top["branch"]),
        "winner_origin": str(top["origin"]),
        "weakest_dimensions": weakest,
        "gate_failures": gate_failures,
        "top_answer_preview": str(top["answer"])[:400],
    }

def repair_contract(contract: NeedSlotContract, omega: Dict[str, Any]) -> NeedSlotContract:
    weak_names = [name for name, _ in omega["weakest_dimensions"]]
    gate_failures = omega["gate_failures"]

    new_boundaries = list(contract.boundary_conditions)
    new_forbidden = set(contract.forbidden_neighbors)

    repair_note = {
        "depth_repair": len(contract.repair_history) + 1,
        "weakest_dimensions": omega["weakest_dimensions"],
        "gate_failures": gate_failures,
    }

    if "prompt" in weak_names or "prompt_fit_ok" in gate_failures:
        new_boundaries.append(
            "repair target: explicitly answer this prompt using its domain carriers and action terms"
        )
    if "anti_boilerplate" in weak_names or "anti_boilerplate_ok" in gate_failures:
        new_boundaries.append(
            "repair target: remove controller boilerplate and replace it with concrete mechanism"
        )
        new_forbidden.add("controller boilerplate")
    if "boundary" in weak_names:
        new_boundaries.append(
            "repair target: name the boundary condition and how the answer respects it"
        )
    if "trap" in weak_names:
        new_boundaries.append(
            "repair target: name the wrong carrier/trap and reject it"
        )
    if "collapse" in weak_names:
        new_boundaries.append(
            "repair target: produce one executable answer and one check"
        )
    if "function" in weak_names:
        new_boundaries.append(
            "repair target: state the preserved operation in concrete terms"
        )

    return replace(
        contract,
        boundary_conditions=new_boundaries,
        forbidden_neighbors=sorted(new_forbidden),
        repair_history=contract.repair_history + [repair_note],
    )


## 8. Recursive resolver

This is the actual KRRB fold:

$$
C_t \xrightarrow{\text{generate}} B_t
\xrightarrow{\text{audit}} A_t
\xrightarrow{\text{gate}} \Psi \text{ or } \Omega_t
\xrightarrow{\text{repair}} C_{t+1}
$$

No parameter-grid fake recursion.


In [9]:
def krrb_recursive_resolve(
    prompt: str,
    contract: NeedSlotContract,
    depth: int = 0,
    trace: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:

    if trace is None:
        trace = []

    candidates = generate_candidates(prompt, contract)
    score_df = score_candidates(prompt, contract, candidates)

    direct = direct_collapse_gate(score_df)
    consensus = consensus_gate(contract, score_df)

    step = {
        "depth": depth,
        "contract": asdict(contract),
        "candidate_origins": score_df[["branch", "origin", "error"]].to_dict(orient="records"),
        "scores": score_df.drop(columns=["detail"]).to_dict(orient="records"),
        "direct_gate": direct,
        "consensus_gate": consensus,
    }
    trace.append(step)

    if direct["ok"]:
        return {
            "state": "Ψ",
            "reason": direct["reason"],
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    if consensus["ok"]:
        return {
            "state": "Ψ",
            "reason": consensus["reason"],
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    omega = extract_omega(score_df, contract)
    step["omega"] = omega

    if depth >= MAX_RECURSION_DEPTH:
        return {
            "state": "Ω",
            "reason": "max_recursion_depth",
            "omega": omega,
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    repaired_contract = repair_contract(contract, omega)

    return krrb_recursive_resolve(
        prompt=prompt,
        contract=repaired_contract,
        depth=depth + 1,
        trace=trace,
    )

def save_json(obj: Dict[str, Any], path: Path):
    def clean(x):
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, pd.DataFrame):
            return x.to_dict(orient="records")
        return str(x)

    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=clean, ensure_ascii=False)

def run_rhi_v12(prompt: str, save: bool = True, show: bool = True) -> Dict[str, Any]:
    contract = build_contract(prompt)
    result = krrb_recursive_resolve(prompt, contract)

    winner = result["winner"]
    final = {
        "run_id": RUN_ID,
        "prompt": prompt,
        "state": result["state"],
        "reason": result["reason"],
        "depth": result["depth"],
        "winner_branch": winner["branch"],
        "winner_origin": winner.get("origin"),
        "winner_error": winner.get("error"),
        "winner_score": winner["score"],
        "answer": winner["answer"],
        "contract": asdict(contract),
        "trace": result["trace"],
        "device_info": DEVICE_INFO,
        "model_ready": MODEL_READY,
        "model_error": MODEL_ERROR,
        "model_id_or_path": MODEL_ID_OR_PATH,
    }

    if save:
        prompt_id = hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:10]
        out_path = OUT_DIR / f"{RUN_ID}_{prompt_id}_result.json"
        rows_path = OUT_DIR / f"{RUN_ID}_{prompt_id}_score_rows.csv"
        save_json(final, out_path)
        result["score_df"].drop(columns=["detail"]).to_csv(rows_path, index=False)
        print("saved:", out_path)
        print("saved:", rows_path)

    if show:
        print("\nSTATE:", final["state"], "| REASON:", final["reason"], "| DEPTH:", final["depth"])
        print("WINNER:", final["winner_branch"], "| ORIGIN:", final["winner_origin"], "| SCORE:", round(float(final["winner_score"]), 4))
        if final["winner_error"]:
            print("WINNER_ERROR:", final["winner_error"])

        print("\nCONTRACT\n--------")
        print(contract_to_text(build_contract(prompt)))

        print("\nFINAL SCORE ROWS\n----------------")
        display(result["score_df"].drop(columns=["detail"]))

        print("\nANSWER\n------")
        print(final["answer"])

        print("\nTRACE SUMMARY\n-------------")
        for step in final["trace"]:
            dg = step["direct_gate"]
            print(
                f"depth={step['depth']} direct={dg['ok']} reason={dg['reason']} "
                f"top={dg.get('top_branch')} checks={dg.get('checks')}"
            )
            if "omega" in step:
                print("  Ω:", step["omega"])

    return final


## 9. Live run

The default prompt is the one that exposed the earlier false-collapse problem.


In [10]:
LIVE_PROMPT = "explain why current AI agents fail when they use tools before forming a contract"

live_result = run_rhi_v12(LIVE_PROMPT.strip(), save=True, show=True)


saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_6e5b59d364_result.json
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_6e5b59d364_score_rows.csv

STATE: Ω | REASON: max_recursion_depth | DEPTH: 3
WINNER: construct | ORIGIN: fallback | SCORE: 0.7988
WINNER_ERROR: AttributeError: 

CONTRACT
--------
ACTIVE_TEMPLATES: CONTRACT, TOOL
INVERSE_NEED: build the missing operational shape demanded by this prompt; answer this prompt specifically; reject generic framework boilerplate
PRESERVED_FUNCTION: form the contract before tool/action selection
DOMAIN_CARRIER: explain, current, ai, agents, fail, use, tools, before, forming
ACTION_TERMS: explain, fail, use, forming
FORBIDDEN_NEIGHBORS: api reflex | generic controller boilerplate | premature execution | surface label | surface task completion | tool-first action
BOUNDARY_CONDITIONS: do not collapse on shared Nexus vocabulary alone | do not collapse on generic controller boilerplate | require prompt-specific d

,branch,origin,score,contract_stance,trace_sufficiency,origin_bonus,quality_grounded_hmean,quality_core_hmean,F_need,F_function,...,action_hit,function_hit,anti_boilerplate,boilerplate_count,branch_leak_penalty,hot,cold,hotcold_balance,error,answer
0,construct,fallback,0.798761,0.708333,1.0,0.25,0.653134,0.578762,0.831250,1.0000,...,1.00,1.0,1.0,0.0,0.0,0.752131,0.678125,0.925994,AttributeError:,Δ Operational answer for this prompt: active=C...
1,verify,fallback,0.717828,0.645833,0.6,0.25,0.657617,0.593677,0.686111,0.9375,...,0.75,1.0,1.0,0.0,0.0,0.693750,0.668750,0.975000,AttributeError:,Verification: test whether the candidate actua...
2,repair,fallback,0.691260,0.625000,0.6,0.25,0.587997,0.516195,0.629861,0.9375,...,0.75,1.0,1.0,0.0,0.0,0.722001,0.631250,0.909249,AttributeError:,"Repair: if prompt grounding is weak, inject th..."
3,counter,fallback,0.672694,0.625000,0.4,0.25,0.603234,0.532650,0.629861,0.8625,...,0.75,1.0,1.0,0.0,0.0,0.640997,0.606250,0.965253,AttributeError:,Wrong path: answering with a generic contract ...



ANSWER
------
Δ Operational answer for this prompt: active=CONTRACT, TOOL. The missing shape is not a noun; it is the operation `form the contract before tool/action selection`. Use carriers [explain, current, ai, agents, fail, use, tools, before] and actions [explain, fail, use, forming] as the fit test. The answer must perform that operation, reject api reflex, and leave a trace showing why the selected path fits.

TRACE SUMMARY
-------------
depth=0 direct=False reason=no_direct_collapse top=construct checks={'score_ok': True, 'margin_ok': True, 'prompt_fit_ok': True, 'anti_boilerplate_ok': True, 'trace_ok': True, 'quality_ok': True, 'leak_ok': True, 'origin_ok': False}
  Ω: {'winner_branch': 'construct', 'winner_origin': 'fallback', 'weakest_dimensions': [('boundary', 0.275), ('quality', 0.6531337923006881), ('trap', 0.6625000000000001)], 'gate_failures': ['origin_ok'], 'top_answer_preview': 'Δ Operational answer for this prompt: active=CONTRACT, TOOL. The missing shape is not a n

## 10. Batch tests

These are the same v11 prompts so the comparison is clean.


In [11]:
TEST_PROMPTS = [
    "explain why current AI agents fail when they use tools before forming a contract",
    "how should a LoRA adapter train a slot-builder without overwriting the base model",
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
    "fix a recursive AI controller that collapses because two branches share vocabulary but disagree operationally",
    "explain memory in an agent as trace continuity rather than a text summary",
]

batch_rows = []
for p in TEST_PROMPTS:
    print("\n" + "=" * 100)
    print("PROMPT:", p)
    res = run_rhi_v12(p, save=True, show=False)
    batch_rows.append({
        "prompt": p,
        "state": res["state"],
        "reason": res["reason"],
        "depth": res["depth"],
        "winner_branch": res["winner_branch"],
        "winner_origin": res["winner_origin"],
        "winner_score": res["winner_score"],
        "winner_error": res["winner_error"],
        "answer_preview": res["answer"][:260].replace("\n", " "),
    })

batch_df = pd.DataFrame(batch_rows)
display(batch_df)

batch_path = OUT_DIR / f"{RUN_ID}_batch_summary.csv"
batch_df.to_csv(batch_path, index=False)
print("saved:", batch_path)



PROMPT: explain why current AI agents fail when they use tools before forming a contract
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_6e5b59d364_result.json
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_6e5b59d364_score_rows.csv

PROMPT: how should a LoRA adapter train a slot-builder without overwriting the base model
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_84a5615f0a_result.json
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_84a5615f0a_score_rows.csv

PROMPT: design a shape-first retrieval step where no noun match exists but the inverse need is clear
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_a8ce36a582_result.json
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_a8ce36a582_score_rows.csv

PROMPT: fix a recursive AI controller that collapses because two branches share vocabulary but disagree operationally
saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_

,prompt,state,reason,depth,winner_branch,winner_origin,winner_score,winner_error,answer_preview
0,explain why current AI agents fail when they u...,Ω,max_recursion_depth,3,construct,fallback,0.798761,AttributeError:,Δ Operational answer for this prompt: active=C...
1,how should a LoRA adapter train a slot-builder...,Ω,max_recursion_depth,3,construct,fallback,0.792208,AttributeError:,Δ Operational answer for this prompt: active=G...
2,design a shape-first retrieval step where no n...,Ω,max_recursion_depth,3,construct,fallback,0.851132,AttributeError:,Δ Operational answer for this prompt: active=G...
3,fix a recursive AI controller that collapses b...,Ω,max_recursion_depth,3,construct,fallback,0.810084,AttributeError:,Δ Operational answer for this prompt: active=R...
4,explain memory in an agent as trace continuity...,Ω,max_recursion_depth,3,construct,fallback,0.795527,AttributeError:,Δ Operational answer for this prompt: active=T...


saved: D:\@User Data\Downloads\rhi_v12_outputs\rhi_v12_a551b6e41e_batch_summary.csv


## 11. What to look for

### Good signs

- `winner_origin = model` for most or all runs.
- Some prompts may resolve at depth `1`, not always `0`.
- `reason = direct_grounded_collapse` only when prompt fit and anti-boilerplate pass.
- `Ω` appears when the answer is generic.
- The batch no longer has the same answer preview for all prompts.

### Bad signs

- Every branch says `origin=fallback`.
- Every prompt still collapses at depth `0`.
- `winner_answer` starts with reusable controller language.
- `F_prompt` is low but direct collapse still passes.
- `anti_boilerplate` is low but direct collapse still passes.

### If you see model AttributeError again

The output now records the exception in `winner_error` and per-candidate `error`.

That means the next patch is not theory. It is generation plumbing: tokenizer chat template/device handling.
